# HybridTCN CL HO RB Optuna

Trains `HybridTCN` on the V3 continuous dataset for `CL`, `HO`, and `RB` using the same high-level flow as the original optimization notebook: data prep, Optuna search, final training, and validation backtest.

Architecture-specific MMTF/PTP diagnostics are intentionally removed.


## 1. Environment Setup

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab - skipping drive mount")

In [ ]:
# Add CTAFlow to path (Colab only)
if IN_COLAB:
    import sys
    %cd /content/drive/MyDrive/CTAEnv/
    %cd CTAFlow
    !git pull
    %cd ..
    sys.path.insert(0, '/content/drive/MyDrive/CTAEnv/CTAFlow/')
    sys.path.insert(1, '/content/drive/MyDrive/CTAEnv/SierraPy')
    !pip install -e SierraPy -q
    !pip install -e CTAFlow -q
    !pip install optuna -q
else:
    print("Running locally - ensure CTAFlow and optuna are installed")

In [ ]:
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Optuna version: {optuna.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Patch torch._utils if missing (some PyTorch builds lack it, breaking _dynamo)
import types
if not hasattr(torch, '_utils'):
    torch._utils = types.ModuleType('torch._utils')
    def _get_device_index(device, optional=False, allow_cpu=False):
        if isinstance(device, int):
            return device
        if isinstance(device, str):
            device = torch.device(device)
        if isinstance(device, torch.device):
            if device.type == 'cpu':
                return -1 if allow_cpu else 0
            return device.index if device.index is not None else 0
        if optional:
            return -1
        raise ValueError(f"Expected device, got {device}")
    torch._utils._get_device_index = _get_device_index
    print("Patched torch._utils (missing in this PyTorch build)")

# Optionally disable dynamo entirely if it keeps causing issues
# torch._dynamo.config.suppress_errors = True

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 2. Configuration

In [ ]:
# --- Tickers ---
TICKERS = ['CL', 'HO', 'RB']

# --- Model ---
MODEL_NAME = 'HybridTCN'
USE_PTP = False
USE_FUSED_SPATIAL = True
USE_AMP = True
SPATIAL_ENCODER = 'fused' if USE_FUSED_SPATIAL else 'separate'

# --- Target ---
TARGET_HORIZON_MINUTES = 60
BAR_MINUTES = 5

# --- Session Filter ---
SAMPLE_SESSION = 'overlap'
SAMPLE_SESSION_START = '06:00'
SAMPLE_SESSION_END = '13:00'

# --- Stride / cached lookbacks ---
SAMPLE_STRIDE = 12
SPATIAL_LOOKBACK_BARS = 8
MAX_TECH_LOOKBACK = 128
MAX_SEQ_LOOKBACK = 64
MAX_NUMBARS_LOOKBACK = 24

# --- AE ---
AE_WINDOW = 21
F_AE = 4
assert AE_WINDOW == 21, 'HybridTCN currently assumes ae_window=21.'

# --- Paths ---
if IN_COLAB:
    DRIVE_PATH = Path('/content/drive/MyDrive')
    DATA_ROOT = DRIVE_PATH / 'features'
    RESULTS_PATH = DRIVE_PATH / 'results' / 'hybrid_tcn_cl_ho_rb'
else:
    _cwd = Path.cwd()
    _default_data_root = Path('/workspace')
    if not _default_data_root.exists():
        _default_data_root = _cwd
    DATA_ROOT = _default_data_root
    RESULTS_PATH = (_cwd / 'notebooks' / 'results' / 'hybrid_tcn_cl_ho_rb').resolve()

RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print(f'Data root: {DATA_ROOT}')
print(f'Results path: {RESULTS_PATH}')
print(f'Tickers: {TICKERS}')
print(f'Target: {TARGET_HORIZON_MINUTES}min forward return')
print(f'Model: {MODEL_NAME}')
print(f'PTP enabled: {USE_PTP}')
print(f'Spatial encoder: {SPATIAL_ENCODER}')
print(f'Stride: {SAMPLE_STRIDE} bars')
print(f'Cached lookbacks: tech={MAX_TECH_LOOKBACK}, seq={MAX_SEQ_LOOKBACK}, spatial={MAX_NUMBARS_LOOKBACK}')


In [ ]:
# Verify data files for each ticker
print("Checking data files...")
base_required_files = ['intraday.csv', 'profiles.npz', 'rasterized.npz', 'vpin.parquet']

all_found = True
for ticker in TICKERS:
    ticker_path = DATA_ROOT / ticker
    print(f"\n{ticker}:")
    required_files = [f'{ticker}_numbars.npz', *base_required_files]
    for fname in required_files:
        fpath = ticker_path / fname
        status = "[OK]" if fpath.exists() else "[MISSING]"
        print(f"  {status} {fname}")
        if not fpath.exists():
            all_found = False

if not all_found:
    print("\n[WARNING] Some files missing - data loading may fail")

## 3. Load Data via V3ContinuousPrep

In [ ]:
from torch.utils.data import DataLoader
from CTAFlow.data.datasets.v3_continuous import (
    V3ContinuousPrep,
    V3ContinuousDataset,
    v3_collate_fn,
    unpack_v3_batch,
)
from CTAFlow.models.prep.intraday_continuous import SessionSpec

print('Loading ticker data...')
prep = V3ContinuousPrep.from_directories(
    root_dir=DATA_ROOT,
    tickers=TICKERS,
    sessions=[SessionSpec('custom', '06:00', '13:00')],
    bar_minutes=BAR_MINUTES,
    target_horizon_minutes=TARGET_HORIZON_MINUTES,
    ae_window=AE_WINDOW,
)

dims = prep.get_dims()
print(f'\nFeature dimensions: {dims}')
for ticker in TICKERS:
    nb_ts, nb_vals = prep._numbars_ts.get(ticker, (np.array([], dtype='datetime64[ns]'), np.empty((0, 4, 32), dtype=np.float32)))
    print(f'{ticker}: number bars loaded={len(nb_ts):,}, number bar tensor shape={nb_vals.shape}')
    assert len(nb_ts) > 0, f'{ticker} missing timestamped NumberBars ({ticker}_numbars.npz)'

print(f'n_tickers: {prep.n_tickers}')
print(f'n_asset_classes: {prep.n_asset_classes}')
print(f'n_asset_subclasses: {prep.n_asset_subclasses}')

print(f'\nPre-building samples (tech={MAX_TECH_LOOKBACK}, seq={MAX_SEQ_LOOKBACK}, nb={MAX_NUMBARS_LOOKBACK})...')
_all_samples = prep.build_samples(
    tech_lookback=MAX_TECH_LOOKBACK,
    seq_lookback_bars=MAX_SEQ_LOOKBACK,
    numbars_lookback=MAX_NUMBARS_LOOKBACK,
    use_fused_spatial=USE_FUSED_SPATIAL,
    session_only=True,
    sample_session=SAMPLE_SESSION,
    sample_session_start=SAMPLE_SESSION_START,
    sample_session_end=SAMPLE_SESSION_END,
    stride=SAMPLE_STRIDE,
)
assert _all_samples, 'No valid samples produced. Check diagnostics above.'

_all_dates = sorted(set(s['date'] for s in _all_samples))
_n_val = max(1, int(len(_all_dates) * 0.2))
VAL_CUTOFF = _all_dates[-_n_val]
TRAIN_SAMPLES = [s for s in _all_samples if s['date'] < VAL_CUTOFF]
VAL_SAMPLES = [s for s in _all_samples if s['date'] >= VAL_CUTOFF]
del _all_samples

print(f'Cached samples: {len(TRAIN_SAMPLES)} train, {len(VAL_SAMPLES)} val (cutoff={VAL_CUTOFF})')

F_TECH = dims['f_tech']
F_SEQ = dims['f_seq']
NUMBARS_CHANNELS = dims['numbars_channels']
VPIN_CHANNELS = dims.get('vpin_channels', 4)
VPIN_BINS = dims.get('vpin_bins', 128)
VPIN_TIME = dims.get('vpin_time', 12)
FUSED_SPATIAL_CHANNELS = NUMBARS_CHANNELS + 3
FUSED_SPATIAL_BINS = prep.numbars_bar_shape[1]

def _truncate_sample(sample, tech_lookback=None, seq_lookback=None, numbars_lookback=None, use_fused_spatial=True):
    s = dict(sample)
    if tech_lookback is not None and s['tech_features'].shape[0] > tech_lookback:
        s['tech_features'] = s['tech_features'][-tech_lookback:]
        s['tech_len'] = tech_lookback
    if seq_lookback is not None and s['seq_vpin'].shape[0] > seq_lookback:
        s['seq_vpin'] = s['seq_vpin'][-seq_lookback:]
        s['seq_vpin_len'] = len(s['seq_vpin'])
    if numbars_lookback is not None:
        if use_fused_spatial and 'fused_spatial' in s and s['fused_spatial'].shape[0] > numbars_lookback:
            s['fused_spatial'] = s['fused_spatial'][-numbars_lookback:]
        elif 'numbars_recent' in s and s['numbars_recent'].shape[0] > numbars_lookback:
            s['numbars_recent'] = s['numbars_recent'][-numbars_lookback:]
    return s

def loaders_from_cached_samples(train_samples, val_samples, prep, batch_size, use_fused_spatial, tech_lookback, seq_lookback, numbars_lookback, shuffle_train=True, num_workers=0):
    train_view = [_truncate_sample(s, tech_lookback, seq_lookback, numbars_lookback, use_fused_spatial) for s in train_samples]
    val_view = [_truncate_sample(s, tech_lookback, seq_lookback, numbars_lookback, use_fused_spatial) for s in val_samples]
    fused_tail_shape = (prep.numbars_bar_shape[0] + 3, prep.numbars_bar_shape[1])
    train_ds = V3ContinuousDataset(train_view, profile_shape=prep.profile_shape, raster_shape=prep.raster_shape, fused_tail_shape=fused_tail_shape)
    val_ds = V3ContinuousDataset(val_view, profile_shape=prep.profile_shape, raster_shape=prep.raster_shape, fused_tail_shape=fused_tail_shape)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=shuffle_train, collate_fn=v3_collate_fn, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=v3_collate_fn, num_workers=num_workers)
    return train_loader, val_loader


In [ ]:
# Verify tech features include tod_sin
assert 'tod_sin' in prep._tech_feature_cols, "tod_sin missing from feature cols!"
print("tod_sin confirmed in feature columns")

# Quick target distribution check
fig, axes = plt.subplots(1, len(TICKERS), figsize=(6*len(TICKERS), 4))
if len(TICKERS) == 1:
    axes = [axes]

for ticker, ax in zip(TICKERS, axes):
    df = prep._tech_dfs.get(ticker)
    if df is None:
        continue
    target = df[prep._target_col].dropna()
    ax.hist(target.values, bins=100, alpha=0.7, color='steelblue')
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'{ticker} Target ({prep._target_col})')
    ax.set_xlabel('30min Forward Log Return')
    ax.set_ylabel('Count')
    print(f"{ticker}: mean={target.mean():.6f}, std={target.std():.6f}, n={len(target)}")

plt.tight_layout()
plt.show()

## 4. Define Optuna Objective

In [ ]:
from CTAFlow.models.deep_learning.multi_branch.tcn import HybridTCN
from CTAFlow.models.deep_learning.multi_branch.tft.mmtf_v3_core import (
    StatefulMMTFv3Core,
    train_epoch_v3_stateful,
    evaluate_v3_stateful,
    ContinuousTradingLoss,
    SharpeScheduler,
)

print('Model input dimensions:')
print(f'  f_tech={F_TECH}')
print(f'  f_seq={F_SEQ}')
print(f'  f_ae={F_AE}')
print(f'  fused_spatial: channels={FUSED_SPATIAL_CHANNELS}, bins={FUSED_SPATIAL_BINS}')
print(f'  vpin_raster: time={VPIN_TIME}, channels={VPIN_CHANNELS}, bins={VPIN_BINS}')


In [ ]:
def objective(trial: optuna.Trial) -> float:
    d_model = trial.suggest_categorical('d_model', [64, 128])
    d_static_emb = trial.suggest_categorical('d_static_emb', [32, 64])
    tcn_depth = trial.suggest_int('tcn_depth', 2, 4)
    kernel_size = trial.suggest_categorical('kernel_size', [2, 3, 5])
    dropout = trial.suggest_float('dropout', 0.1, 0.4)
    seq_layers = trial.suggest_int('seq_layers', 1, 3)
    seq_nheads = trial.suggest_categorical('seq_nheads', [2, 4])
    d_latent = trial.suggest_categorical('d_latent', [32, 64])
    d_ae_hidden = trial.suggest_categorical('d_ae_hidden', [64, 128])
    kl_weight = trial.suggest_float('kl_weight', 0.001, 0.1, log=True)
    recon_weight = trial.suggest_float('recon_weight', 0.01, 0.5, log=True)
    regime_floor = trial.suggest_float('regime_floor', 0.05, 0.3)

    state_hidden_dim = trial.suggest_categorical('state_hidden_dim', [8, 16, 32])
    state_momentum = trial.suggest_float('state_momentum', 0.90, 0.99)

    tech_lookback = trial.suggest_categorical('tech_lookback', [48, 96, 128])
    seq_lookback = trial.suggest_categorical('seq_lookback', [16, 32, 48, 64])
    numbars_lookback = trial.suggest_categorical('numbars_lookback', [8, 12, 16, 24])
    batch_size = trial.suggest_categorical('batch_size', [48, 64, 96])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-3, 7e-3, log=True)
    max_norm = trial.suggest_float('max_norm', 0.5, 1.0)

    tc_cost = trial.suggest_float('tc_cost', 5e-5, 5e-4, log=True)
    init_direction_weight = trial.suggest_float('init_direction_weight', 0.5, 1.5)
    final_direction_weight = trial.suggest_float('final_direction_weight', 0.05, 0.3)
    init_reg_weight = trial.suggest_float('init_reg_weight', 0.1, 0.5)
    target_exposure = trial.suggest_float('target_exposure', 0.2, 0.5)
    downside_vol_weight = trial.suggest_float('downside_vol_weight', 0.02, 0.75, log=True)
    holding_weight = trial.suggest_float('holding_weight', 0.0, 0.5)
    exposure_asymmetry = trial.suggest_float('exposure_asymmetry', 1.0, 6.0)

    num_epochs = 20
    warmup_epochs = 5

    try:
        train_loader, val_loader = loaders_from_cached_samples(
            TRAIN_SAMPLES, VAL_SAMPLES, prep,
            batch_size=batch_size,
            use_fused_spatial=USE_FUSED_SPATIAL,
            tech_lookback=tech_lookback,
            seq_lookback=seq_lookback,
            numbars_lookback=numbars_lookback,
        )
    except Exception as e:
        print(f'Dataloader failed: {e}')
        return -1e9

    base_model = HybridTCN(
        f_tech=F_TECH, f_seq=F_SEQ, f_ae=F_AE,
        d_latent=d_latent, d_ae_hidden=d_ae_hidden,
        kl_weight=kl_weight, recon_weight=recon_weight,
        numbars_channels=NUMBARS_CHANNELS,
        vpin_channels=VPIN_CHANNELS, vpin_bins=VPIN_BINS, vpin_time=VPIN_TIME,
        n_tickers=prep.n_tickers, n_asset_classes=prep.n_asset_classes, n_asset_subclasses=prep.n_asset_subclasses,
        d_model=d_model, d_static_emb=d_static_emb,
        tcn_channels=[d_model] * tcn_depth, kernel_size=kernel_size,
        spatial_encoder=SPATIAL_ENCODER, seq_layers=seq_layers, seq_nheads=seq_nheads,
        regime_gate=True, regime_floor=regime_floor, dropout=dropout,
    )
    model = StatefulMMTFv3Core(
        base_model=base_model,
        n_tickers=prep.n_tickers,
        quantile_head=False,
        state_hidden_dim=state_hidden_dim,
        state_momentum=state_momentum,
        update_on_eval=True,
    ).to(device)

    loss_fn = ContinuousTradingLoss(
        tc_cost=tc_cost,
        direction_weight=init_direction_weight,
        reg_weight=init_reg_weight,
        target_exposure=target_exposure,
        use_sortino=True,
        downside_vol_weight=downside_vol_weight,
        tc_in_sharpe=True,
        holding_weight=holding_weight,
        exposure_asymmetry=exposure_asymmetry,
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    sharpe_sched = SharpeScheduler(
        warmup_epochs=warmup_epochs, total_epochs=num_epochs,
        initial_direction_weight=init_direction_weight, final_direction_weight=final_direction_weight,
        initial_reg_weight=init_reg_weight, final_reg_weight=0.05,
        initial_target_exposure=0.2, final_target_exposure=target_exposure,
        initial_holding_weight=0.0, final_holding_weight=holding_weight,
    )
    scaler = torch.amp.GradScaler() if (USE_AMP and device.type == 'cuda') else None

    best_sharpe = -1e9
    patience_counter = 0
    prev_val_loss = None

    for epoch in range(num_epochs):
        sharpe_sched.step(epoch, loss_fn)
        is_warmup = epoch < warmup_epochs
        train_loss, train_metrics = train_epoch_v3_stateful(model, train_loader, loss_fn, optimizer, device, max_norm=max_norm, unpack_fn=unpack_v3_batch, scaler=scaler)
        val_metrics = evaluate_v3_stateful(model, val_loader, loss_fn, device, unpack_fn=unpack_v3_batch)
        scheduler.step()

        val_loss = val_metrics['loss']
        val_sharpe = val_metrics['sharpe']
        if math.isnan(val_loss) or math.isinf(val_loss) or val_loss > 100.0:
            return best_sharpe if best_sharpe > -1e9 else -1e9
        if prev_val_loss is not None and epoch >= 3 and val_loss > abs(prev_val_loss) * 5.0:
            return best_sharpe if best_sharpe > -1e9 else -1e9
        prev_val_loss = val_loss

        print(f'  E{epoch+1:02d} | Loss: {val_loss:.4f} | Sharpe: {val_sharpe:.4f} | WinRate: {val_metrics["win_rate"]:.1f}% | DirAcc: {val_metrics["dir_accuracy"]:.1f}% | Exposure: {val_metrics["avg_exposure"]:.3f}{" [warmup]" if is_warmup else ""}')

        if is_warmup:
            continue
        if val_sharpe > best_sharpe:
            best_sharpe = val_sharpe
            patience_counter = 0
            trial.set_user_attr('final_sharpe', val_sharpe)
            trial.set_user_attr('final_sortino', val_metrics['sortino'])
            trial.set_user_attr('final_win_rate', val_metrics['win_rate'])
            trial.set_user_attr('final_dir_acc', val_metrics['dir_accuracy'])
            trial.set_user_attr('final_pf', val_metrics['profit_factor'])
            trial.set_user_attr('final_exposure', val_metrics['avg_exposure'])
            trial.set_user_attr('final_loss', val_loss)
            trial.set_user_attr('final_downside_vol', val_metrics.get('downside_vol', 0.0))
            trial.set_user_attr('state_momentum', state_momentum)
            trial.set_user_attr('state_hidden_dim', state_hidden_dim)
            trial.set_user_attr('tcn_receptive_field', base_model.tcn.receptive_field)
        else:
            patience_counter += 1

        trial.report(val_sharpe, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        if patience_counter >= 8:
            break

    del model, base_model, optimizer, loss_fn, train_loader, val_loader
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return best_sharpe


## 5. Run Optuna Optimization

In [ ]:
N_TRIALS = 30
STUDY_NAME = f'hybridtcn_{"_".join(TICKERS).lower()}_continuous'

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=5),
)

print(f'Starting optimization: {N_TRIALS} trials')
print(f'Study: {STUDY_NAME}')
print(f'Tickers: {", ".join(TICKERS)}')
print(f'Target: {TARGET_HORIZON_MINUTES}min forward return -> continuous position')
print('Head: tanh continuous')
print('Loss: ContinuousTradingLoss + SharpeScheduler')
print('-' * 60)


In [ ]:
study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True,
)

In [ ]:
# Best trial results
best_trial = study.best_trial
print(f'\nBest trial #{best_trial.number}:')
print(f'  Sharpe: {best_trial.value:.6f}')
if best_trial.user_attrs:
    attr_keys = ['final_sharpe', 'final_sortino', 'final_win_rate', 'final_dir_acc', 'final_pf', 'final_exposure', 'final_downside_vol']
    for k in attr_keys:
        print(f'  {k}: {best_trial.user_attrs.get(k, "N/A")}')

print('  Params:')
best_params = best_trial.params
for key, value in sorted(best_params.items()):
    print(f'    {key}: {value}')

best_params['best_value'] = best_trial.value
best_params['tickers'] = TICKERS
best_params['target_horizon_minutes'] = TARGET_HORIZON_MINUTES
best_params['model_name'] = MODEL_NAME
prefix = f'{"_".join(TICKERS).lower()}_hybridtcn_optuna'

with open(RESULTS_PATH / f'{prefix}_best_params.json', 'w', encoding='utf-8') as f:
    json.dump(best_params, f, indent=2)

with open(RESULTS_PATH / f'{prefix}_study.pkl', 'wb') as f:
    pickle.dump(study, f)


In [ ]:
# Save study artifacts
import joblib

joblib.dump(study, RESULTS_PATH / f"{prefix}_study.pkl")
df_trials = study.trials_dataframe()
df_trials.to_csv(RESULTS_PATH / f"{prefix}_all_trials.csv", index=False)
print(f"Study artifacts saved to {RESULTS_PATH}")

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

valid_trials = df_trials[df_trials['state'] == 'COMPLETE']

# 1. Optimization history
ax = axes[0, 0]
ax.plot(valid_trials.index, valid_trials['value'], 'b-o', alpha=0.6, label='Trial Sharpe')
ax.axhline(y=study.best_value, color='r', linestyle='--', label=f'Best: {study.best_value:.4f}')
ax.set_xlabel('Trial')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Optimization History')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Parameter importance
ax = axes[0, 1]
try:
    importances = optuna.importance.get_param_importances(study)
    params = list(importances.keys())[:10]
    values = [importances[p] for p in params]
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(params)))
    ax.barh(params, values, color=colors)
    ax.set_xlabel('Importance')
    ax.set_title('Hyperparameter Importance')
    ax.grid(True, alpha=0.3, axis='x')
except:
    ax.text(0.5, 0.5, 'Not enough completed trials', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Hyperparameter Importance')

# 3. Learning rate vs Sharpe
ax = axes[1, 0]
if 'params_learning_rate' in valid_trials.columns:
    ax.scatter(valid_trials['params_learning_rate'], valid_trials['value'],
               c=valid_trials.index, cmap='viridis', alpha=0.7, s=100)
    ax.set_xscale('log')
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('Sharpe')
    ax.set_title('Learning Rate vs Sharpe')
    ax.grid(True, alpha=0.3)

# 4. d_model vs Sharpe
ax = axes[1, 1]
if 'params_d_model' in valid_trials.columns:
    d_models = sorted(valid_trials['params_d_model'].unique())
    data_by_d = [valid_trials[valid_trials['params_d_model'] == d]['value'].values for d in d_models]
    bp = ax.boxplot(data_by_d, positions=range(len(d_models)), patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Set2(np.linspace(0, 1, len(d_models)))):
        patch.set_facecolor(color)
    ax.set_xticks(range(len(d_models)))
    ax.set_xticklabels([str(int(d)) for d in d_models])
    ax.set_xlabel('d_model')
    ax.set_ylabel('Sharpe')
    ax.set_title('Model Size vs Sharpe')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f"MMTFv3 Optimization ({', '.join(TICKERS)})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_results.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Loss component analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. tc_cost vs Sharpe
ax = axes[0, 0]
if 'params_tc_cost' in valid_trials.columns:
    ax.scatter(valid_trials['params_tc_cost'], valid_trials['value'],
               c=valid_trials.index, cmap='viridis', alpha=0.7, s=80)
    ax.set_xscale('log')
    ax.set_xlabel('Transaction Cost')
    ax.set_ylabel('Sharpe')
    ax.set_title('TC Cost vs Sharpe')
    ax.grid(True, alpha=0.3)

# 2. target_exposure vs Sharpe
ax = axes[0, 1]
if 'params_target_exposure' in valid_trials.columns:
    ax.scatter(valid_trials['params_target_exposure'], valid_trials['value'],
               c=valid_trials.index, cmap='plasma', alpha=0.7, s=80)
    ax.set_xlabel('Target Exposure')
    ax.set_ylabel('Sharpe')
    ax.set_title('Target Exposure vs Sharpe')
    ax.grid(True, alpha=0.3)

# 3. init_direction_weight vs Sharpe
ax = axes[0, 2]
if 'params_init_direction_weight' in valid_trials.columns:
    ax.scatter(valid_trials['params_init_direction_weight'], valid_trials['value'],
               c=valid_trials.index, cmap='coolwarm', alpha=0.7, s=80)
    ax.set_xlabel('Initial Direction Weight')
    ax.set_ylabel('Sharpe')
    ax.set_title('Direction Weight vs Sharpe')
    ax.grid(True, alpha=0.3)

# 4. tech_lookback vs Sharpe
ax = axes[1, 0]
if 'params_tech_lookback' in valid_trials.columns:
    lbs = sorted(valid_trials['params_tech_lookback'].unique())
    data_by_lb = [valid_trials[valid_trials['params_tech_lookback'] == lb]['value'].values for lb in lbs]
    bp = ax.boxplot(data_by_lb, positions=range(len(lbs)), patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Set3(np.linspace(0, 1, len(lbs)))):
        patch.set_facecolor(color)
    ax.set_xticks(range(len(lbs)))
    ax.set_xticklabels([str(int(lb)) for lb in lbs])
    ax.set_xlabel('Tech Lookback (bars)')
    ax.set_ylabel('Sharpe')
    ax.set_title('Tech Lookback vs Sharpe')
    ax.grid(True, alpha=0.3, axis='y')

# 5. Dropout vs Sharpe
ax = axes[1, 1]
if 'params_dropout' in valid_trials.columns:
    ax.scatter(valid_trials['params_dropout'], valid_trials['value'],
               c=valid_trials.index, cmap='plasma', alpha=0.7, s=80)
    ax.set_xlabel('Dropout')
    ax.set_ylabel('Sharpe')
    ax.set_title('Dropout vs Sharpe')
    ax.grid(True, alpha=0.3)

# 6. Backbone comparison
ax = axes[1, 2]
if 'params_backbone' in valid_trials.columns:
    for bb_name in ['mamba', 'transformer']:
        mask = valid_trials['params_backbone'] == bb_name
        if mask.any():
            vals = valid_trials.loc[mask, 'value']
            ax.hist(vals, bins=15, alpha=0.5, label=bb_name.upper())
    ax.set_xlabel('Sharpe')
    ax.set_ylabel('Count')
    ax.set_title('Backbone Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Parameter Analysis ({', '.join(TICKERS)})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_param_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

## 7. Train Final Model with Best Parameters

In [ ]:
best = best_params
print("Training final model with best parameters:")
for k, v in sorted(best.items()):
    if k not in ('best_value', 'tickers', 'target_horizon_minutes', 'backbone'):
        print(f"  {k}: {v}")

In [ ]:
# Build final dataloaders + model
tech_lookback = best['tech_lookback']
seq_lookback = best['seq_lookback']
numbars_lookback = best['numbars_lookback']

train_loader, val_loader = loaders_from_cached_samples(
    TRAIN_SAMPLES, VAL_SAMPLES, prep,
    batch_size=best['batch_size'],
    use_fused_spatial=USE_FUSED_SPATIAL,
    tech_lookback=tech_lookback,
    seq_lookback=seq_lookback,
    numbars_lookback=numbars_lookback,
)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

base_model = HybridTCN(
    f_tech=F_TECH, f_seq=F_SEQ, f_ae=F_AE,
    d_latent=best['d_latent'], d_ae_hidden=best['d_ae_hidden'],
    kl_weight=best['kl_weight'], recon_weight=best['recon_weight'],
    numbars_channels=NUMBARS_CHANNELS, vpin_channels=VPIN_CHANNELS, vpin_bins=VPIN_BINS, vpin_time=VPIN_TIME,
    n_tickers=prep.n_tickers, n_asset_classes=prep.n_asset_classes, n_asset_subclasses=prep.n_asset_subclasses,
    d_model=best['d_model'], d_static_emb=best['d_static_emb'],
    tcn_channels=[best['d_model']] * best['tcn_depth'], kernel_size=best['kernel_size'],
    spatial_encoder=SPATIAL_ENCODER, seq_layers=best['seq_layers'], seq_nheads=best['seq_nheads'],
    regime_gate=True, regime_floor=best['regime_floor'], dropout=best['dropout'],
)

final_model = StatefulMMTFv3Core(
    base_model=base_model, n_tickers=prep.n_tickers, quantile_head=False,
    state_hidden_dim=best['state_hidden_dim'], state_momentum=best['state_momentum'], update_on_eval=True,
).to(device)

print(f'Model parameters: {sum(p.numel() for p in final_model.parameters()):,}')
print(f'TCN receptive field: {base_model.tcn.receptive_field}')


In [ ]:
# Forward-pass sanity check on real data
batch = next(iter(train_loader))
sanity_inputs, sanity_targets = unpack_v3_batch(batch, device=device)
with torch.no_grad():
    position, ae_losses, tracker = final_model(**sanity_inputs, return_ae_losses=True, return_tracker=True)

print(f'position shape: {tuple(position.shape)}')
print(f'target shape:   {tuple(sanity_targets.shape)}')
print(f'AE losses keys: {list(ae_losses.keys())}')
print(f'tracker keys:   {sorted(tracker.keys())}')
if 'fused_spatial' in sanity_inputs:
    print(f'fused_spatial batch shape: {tuple(sanity_inputs["fused_spatial"].shape)}')
print(f'seq_vpin batch shape:      {tuple(sanity_inputs["seq_vpin"].shape)}')


In [ ]:
# Branch timestamp audit: verify all modal inputs are strictly earlier than the anchor.
def _normalize_dt_index(df):
    if df is None or len(df) == 0:
        return pd.DataFrame() if df is None else df.copy()
    out = df.copy()
    if not isinstance(out.index, pd.DatetimeIndex):
        out.index = pd.to_datetime(out.index)
    if out.index.tz is not None:
        out.index = out.index.tz_localize(None)
    out = out.sort_index()
    out = out[~out.index.duplicated(keep='last')]
    return out

def audit_branch_causality(prep, ticker, tech_lookback, seq_lookback_bars, numbars_lookback,
                           sample_session=None, sample_session_start=None, sample_session_end=None):
    df = prep._tech_dfs[ticker].copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    df = df.sort_index()

    bar_times = df.index.time
    if sample_session_start is not None and sample_session_end is not None:
        t_start = pd.Timestamp(sample_session_start).time()
        t_end = pd.Timestamp(sample_session_end).time()
        session_mask = np.array([(t >= t_start) and (t <= t_end) for t in bar_times], dtype=bool)
    elif sample_session is not None:
        col_map = {'usa': 'is_usa', 'london': 'is_london', 'overlap': 'is_session_overlap'}
        session_mask = df[col_map[sample_session.lower()]].astype(bool).values
    elif 'is_active' in df.columns:
        session_mask = df['is_active'].astype(bool).values
    else:
        session_mask = np.ones(len(df), dtype=bool)

    target_arr = df[prep._target_col].values.astype(np.float32)
    eligible = [i for i in range(tech_lookback, len(df)) if session_mask[i] and not np.isnan(target_arr[i])]
    assert eligible, f'No eligible anchor bars for {ticker}'
    bar_idx = eligible[-1]
    anchor_ts = pd.Timestamp(df.index[bar_idx]).tz_localize(None) if pd.Timestamp(df.index[bar_idx]).tzinfo else pd.Timestamp(df.index[bar_idx])
    anchor64 = np.datetime64(anchor_ts, 'ns')

    tech_start = bar_idx - tech_lookback
    tech_ts = df.index[tech_start:bar_idx]
    tech_max = pd.Timestamp(tech_ts[-1]) if len(tech_ts) else None

    nb_ts_arr, _ = prep._numbars_ts.get(ticker, (np.array([], dtype='datetime64[ns]'), None))
    nb_cut = np.searchsorted(nb_ts_arr, anchor64, side='left')
    nb_slice = nb_ts_arr[max(0, nb_cut - numbars_lookback):nb_cut] if nb_cut > 0 else np.array([], dtype='datetime64[ns]')
    nb_max = pd.Timestamp(nb_slice[-1]) if len(nb_slice) else None

    seq_df = _normalize_dt_index(prep._seq_vpin.get(ticker, pd.DataFrame()))
    seq_ts = seq_df.index.values if not seq_df.empty else np.array([], dtype='datetime64[ns]')
    seq_cut = np.searchsorted(seq_ts, anchor64, side='left')
    seq_slice = seq_ts[max(0, seq_cut - seq_lookback_bars):seq_cut] if seq_cut > 0 else np.array([], dtype='datetime64[ns]')
    seq_max = pd.Timestamp(seq_slice[-1]) if len(seq_slice) else None

    spatial_df = _normalize_dt_index(prep._vpin_spatial.get(ticker, pd.DataFrame()))
    spatial_ts = spatial_df.index.values if not spatial_df.empty else np.array([], dtype='datetime64[ns]')
    spatial_cut = np.searchsorted(spatial_ts, anchor64, side='left')
    spatial_slice = spatial_ts[:spatial_cut] if spatial_cut > 0 else np.array([], dtype='datetime64[ns]')
    spatial_max = pd.Timestamp(spatial_slice[-1]) if len(spatial_slice) else None

    print(f'[{ticker}] anchor_ts={anchor_ts}')
    print(f'  tech window max ts:        {tech_max}')
    print(f'  NumberBars max ts:         {nb_max}')
    print(f'  seq_vpin max ts:           {seq_max}')
    print(f'  fused VPIN source max ts:  {spatial_max}')

    assert tech_max is None or tech_max < anchor_ts, f'tech leak: {tech_max} !< {anchor_ts}'
    assert nb_max is None or nb_max < anchor_ts, f'NumberBars leak: {nb_max} !< {anchor_ts}'
    assert seq_max is None or seq_max < anchor_ts, f'seq_vpin leak: {seq_max} !< {anchor_ts}'
    assert spatial_max is None or spatial_max < anchor_ts, f'fused VPIN leak: {spatial_max} !< {anchor_ts}'
    return {
        'ticker': ticker,
        'anchor_ts': anchor_ts,
        'tech_max': tech_max,
        'numbars_max': nb_max,
        'seq_vpin_max': seq_max,
        'fused_vpin_max': spatial_max,
    }

audit_rows = []
for ticker in TICKERS:
    audit_rows.append(
        audit_branch_causality(
            prep,
            ticker=ticker,
            tech_lookback=tech_lookback,
            seq_lookback_bars=seq_lookback,
            numbars_lookback=numbars_lookback,
            sample_session=SAMPLE_SESSION,
            sample_session_start=SAMPLE_SESSION_START,
            sample_session_end=SAMPLE_SESSION_END,
        )
    )

audit_df = pd.DataFrame(audit_rows)
audit_df


In [ ]:
NUM_EPOCHS = 30
WARMUP_EPOCHS = 5

loss_fn = ContinuousTradingLoss(
    tc_cost=best['tc_cost'],
    direction_weight=best['init_direction_weight'],
    reg_weight=best['init_reg_weight'],
    target_exposure=best['target_exposure'],
    use_sortino=True,
    downside_vol_weight=best['downside_vol_weight'],
    tc_in_sharpe=True,
    holding_weight=best['holding_weight'],
    exposure_asymmetry=best['exposure_asymmetry'],
).to(device)
optimizer = optim.AdamW(final_model.parameters(), lr=best['learning_rate'], weight_decay=best['weight_decay'])
lr_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=best['learning_rate'] * 0.001)
sharpe_sched = SharpeScheduler(
    warmup_epochs=WARMUP_EPOCHS, total_epochs=NUM_EPOCHS,
    initial_direction_weight=best['init_direction_weight'], final_direction_weight=best['final_direction_weight'],
    initial_reg_weight=best['init_reg_weight'], final_reg_weight=0.05,
    initial_target_exposure=0.2, final_target_exposure=best['target_exposure'],
    initial_holding_weight=0.0, final_holding_weight=best['holding_weight'],
)

history = {'train_loss': [], 'val_loss': [], 'val_sharpe': [], 'val_sortino': [], 'val_win_rate': [], 'val_dir_acc': [], 'val_pf': [], 'val_exposure': [], 'val_max_dd': [], 'val_downside_vol': [], 'lr': [], 'direction_weight': [], 'reg_weight': []}
best_sharpe = -1e9
best_state = None
scaler = torch.amp.GradScaler() if (USE_AMP and device.type == 'cuda') else None

print(f'Training for {NUM_EPOCHS} epochs ({MODEL_NAME} + StateLayer)')
print('=' * 100)
for epoch in range(NUM_EPOCHS):
    sharpe_sched.step(epoch, loss_fn)
    train_loss, train_metrics = train_epoch_v3_stateful(final_model, train_loader, loss_fn, optimizer, device, max_norm=best['max_norm'], unpack_fn=unpack_v3_batch, scaler=scaler)
    val_metrics = evaluate_v3_stateful(final_model, val_loader, loss_fn, device, unpack_fn=unpack_v3_batch)
    lr_scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['val_sharpe'].append(val_metrics['sharpe'])
    history['val_sortino'].append(val_metrics['sortino'])
    history['val_win_rate'].append(val_metrics['win_rate'])
    history['val_dir_acc'].append(val_metrics['dir_accuracy'])
    history['val_pf'].append(val_metrics['profit_factor'])
    history['val_exposure'].append(val_metrics['avg_exposure'])
    history['val_max_dd'].append(val_metrics['max_drawdown'])
    history['val_downside_vol'].append(val_metrics.get('downside_vol', 0.0))
    history['lr'].append(optimizer.param_groups[0]['lr'])
    history['direction_weight'].append(loss_fn.direction_weight)
    history['reg_weight'].append(loss_fn.reg_weight)

    is_best = val_metrics['sharpe'] > best_sharpe
    if is_best:
        best_sharpe = val_metrics['sharpe']
        best_state = copy.deepcopy(final_model.state_dict())

    print(f'E{epoch+1:02d}/{NUM_EPOCHS} | Loss: {train_loss:.4f}/{val_metrics["loss"]:.4f} | Sharpe: {val_metrics["sharpe"]:.4f} | Sortino: {val_metrics["sortino"]:.4f} | WR: {val_metrics["win_rate"]:.1f}% | PF: {val_metrics["profit_factor"]:.2f} | Exp: {val_metrics["avg_exposure"]:.3f}{" [BEST]" if is_best else ""}')

if best_state is not None:
    final_model.load_state_dict(best_state)
print('=' * 100)
print(f'Best Sharpe: {best_sharpe:.6f}')


In [ ]:
# Load best model state
if best_state:
    final_model.load_state_dict(best_state)
    if hasattr(final_model, 'reset_position_state'):
        final_model.reset_position_state()
    print("Loaded best model state")



In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].plot(history['train_loss'], label='Train')
axes[0, 0].plot(history['val_loss'], label='Val')
axes[0, 0].set_title('Loss')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history['val_sharpe'], label='Sharpe')
axes[0, 1].plot(history['val_sortino'], label='Sortino')
axes[0, 1].axhline(0, color='gray', linestyle=':')
axes[0, 1].set_title('Risk Metrics')
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(history['val_win_rate'], label='Win Rate')
axes[0, 2].plot(history['val_dir_acc'], label='Dir Accuracy')
axes[0, 2].axhline(50, color='gray', linestyle=':')
axes[0, 2].set_title('Accuracy Metrics')
axes[0, 2].legend(); axes[0, 2].grid(True, alpha=0.3)

axes[1, 0].plot(history['val_pf'])
axes[1, 0].axhline(1.0, color='gray', linestyle=':')
axes[1, 0].set_title('Profit Factor')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history['val_exposure'], label='Avg Exposure')
axes[1, 1].plot(history['val_max_dd'], label='Max Drawdown')
axes[1, 1].plot(history['val_downside_vol'], label='Downside Vol')
axes[1, 1].set_title('Exposure and Risk')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(history['lr'], label='LR')
axes[1, 2].plot(history['direction_weight'], label='Direction Weight')
axes[1, 2].plot(history['reg_weight'], label='Reg Weight')
axes[1, 2].set_yscale('log')
axes[1, 2].set_title('Scheduler State')
axes[1, 2].legend(); axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_PATH / f'{prefix}_training_history.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Model Diagnostics

In [ ]:
# Final metrics and tracker snapshot
final_model.eval()
if hasattr(final_model, 'reset_position_state'):
    final_model.reset_position_state()

with torch.no_grad():
    for batch in val_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        position, ae_losses, tracker = final_model(**inputs, return_ae_losses=True, return_tracker=True)
        break

final_metrics = evaluate_v3_stateful(final_model, val_loader, loss_fn, device, unpack_fn=unpack_v3_batch)
print('Tracker:')
for k, v in tracker.items():
    print(f'  {k}: {v}')
print('\nFinal metrics:')
for k, v in final_metrics.items():
    if isinstance(v, (int, float, np.floating)):
        print(f'  {k}: {v}')


In [ ]:
# Position distribution analysis
final_model.eval()
if hasattr(final_model, 'reset_position_state'):
    final_model.reset_position_state()

all_positions = []
all_returns = []
with torch.no_grad():
    for batch in val_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        position, _ = final_model(**inputs, return_ae_losses=True)
        all_positions.append(position.view(-1).cpu().numpy())
        all_returns.append(targets.view(-1).cpu().numpy())

positions = np.concatenate(all_positions)
returns = np.concatenate(all_returns)
strategy_ret = positions * returns
cum_pnl = np.cumsum(strategy_ret)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(positions, bins=100, alpha=0.7, color='steelblue')
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title(f'Position Distribution (mean={positions.mean():.3f}, std={positions.std():.3f})')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(returns, positions, alpha=0.05, s=5, c='steelblue')
axes[1].axhline(y=0, color='gray', linestyle=':')
axes[1].axvline(x=0, color='gray', linestyle=':')
axes[1].set_title('Position vs Return')
axes[1].grid(True, alpha=0.3)

axes[2].plot(cum_pnl, 'b-', alpha=0.8)
axes[2].axhline(y=0, color='gray', linestyle=':')
axes[2].set_title('Validation Cumulative PnL')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_PATH / f'{prefix}_position_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Save Final Model

In [ ]:
model_path = RESULTS_PATH / f'{prefix}_best_model.pth'

save_dict = {
    'model_state_dict': final_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_sharpe': best_sharpe,
    'params': best,
    'tickers': TICKERS,
    'dims': dims,
    'n_tickers': prep.n_tickers,
    'n_asset_classes': prep.n_asset_classes,
    'n_asset_subclasses': prep.n_asset_subclasses,
    'target_horizon_minutes': TARGET_HORIZON_MINUTES,
    'history': history,
    'final_metrics': final_metrics,
    'architecture': 'HybridTCN+StateLayer',
    'use_ptp': False,
    'state_config': {'state_hidden_dim': best['state_hidden_dim'], 'state_momentum': best['state_momentum'], 'update_on_eval': True},
    'ae_config': {'f_ae': F_AE, 'ae_window': AE_WINDOW, 'd_latent': best['d_latent'], 'd_ae_hidden': best['d_ae_hidden'], 'kl_weight': best['kl_weight'], 'recon_weight': best['recon_weight']},
    'tech_feature_cols': prep._tech_feature_cols,
}

torch.save(save_dict, model_path)
history_df = pd.DataFrame(history)
history_df.to_csv(RESULTS_PATH / f'{prefix}_training_history.csv', index=False)

print('TRAINING COMPLETE')
print('Architecture: HybridTCN + StateLayer')
print(f'Best Sharpe: {best_sharpe:.6f}')
print(f'Artifacts saved to: {RESULTS_PATH}')


## 10. Validation Backtest

In [ ]:
# Validation Backtest
final_model.eval()
if hasattr(final_model, 'reset_position_state'):
    final_model.reset_position_state()

_bt_pos, _bt_ret, _bt_tid = [], [], []
with torch.no_grad():
    for batch in val_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        position, _ = final_model(**inputs, return_ae_losses=True)
        _bt_pos.append(position.view(-1).cpu())
        _bt_ret.append(targets.view(-1).float().cpu())
        _bt_tid.append(inputs['ticker_id'].view(-1).cpu())

bt_pos = torch.cat(_bt_pos).numpy()
bt_ret = torch.cat(_bt_ret).numpy()
bt_tid = torch.cat(_bt_tid).numpy()
id_to_ticker = {meta.ticker_id: t for t, meta in prep.registry.items()}

_TRADE_THRESH = 0.05
def _bt_stats(pos, ret, label):
    sr = pos * ret
    cum = np.cumsum(sr)
    non_flat = np.abs(pos) > _TRADE_THRESH
    gp = sr[sr > 0].sum()
    gl = np.abs(sr[sr < 0]).sum() + 1e-9
    correct = ((pos > 0) & (ret > 0)) | ((pos < 0) & (ret < 0))
    dir_acc = (correct & non_flat).sum() / max(non_flat.sum(), 1)
    win_rate = ((sr[non_flat] > 0).mean() * 100) if non_flat.sum() > 0 else 0.0
    run_max = np.maximum.accumulate(cum)
    mdd = float((run_max - cum).max())
    mean_sr = sr.mean()
    std_sr = sr.std() + 1e-8
    neg_sr = sr[sr < 0]
    dside = float(np.sqrt((neg_sr ** 2).mean())) if len(neg_sr) > 0 else 1e-8
    signs = np.sign(pos)
    n_trades = int((np.diff(signs) != 0).sum())
    return {'Ticker': label, 'Net PnL': round(float(sr.sum()), 6), 'Sharpe': round(float(mean_sr / std_sr), 4), 'Sortino': round(float(mean_sr / (dside + 1e-8)), 4), 'Win Rate (%)': round(float(win_rate), 2), 'Dir Acc (%)': round(float(dir_acc * 100), 2), 'Profit Factor': round(float(gp / gl), 4), 'Max Drawdown': round(mdd, 6), '# Trades': n_trades, '# Active Bars': int(non_flat.sum()), 'Avg |Position|': round(float(np.abs(pos).mean()), 4), 'N Samples': len(pos)}

rows = []
for tid in sorted(id_to_ticker):
    ticker = id_to_ticker[tid]
    mask = bt_tid == tid
    if mask.sum() == 0:
        continue
    rows.append(_bt_stats(bt_pos[mask], bt_ret[mask], ticker))
rows.append(_bt_stats(bt_pos, bt_ret, 'COMBINED'))

df_bt = pd.DataFrame(rows).set_index('Ticker')
print(df_bt.to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
comb_pnl = np.cumsum(bt_pos * bt_ret)
for tid in sorted(id_to_ticker):
    ticker = id_to_ticker[tid]
    mask = bt_tid == tid
    axes[0].plot(np.cumsum((bt_pos * bt_ret)[mask]), label=ticker, alpha=0.85)
axes[0].plot(comb_pnl, label='Combined', color='black', linestyle='--', linewidth=2)
axes[0].axhline(0, color='gray', linestyle=':')
axes[0].set_title('Cumulative PnL')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

_tickers_only = [l for l in df_bt.index if l != 'COMBINED']
_sharpe_vals = [df_bt.loc[t, 'Sharpe'] for t in _tickers_only]
_cols = ['#27ae60' if v >= 0 else '#e74c3c' for v in _sharpe_vals]
axes[1].bar(_tickers_only, _sharpe_vals, color=_cols, alpha=0.85)
axes[1].axhline(0, color='gray', linestyle=':')
axes[1].set_title('Per-Ticker Sharpe')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(RESULTS_PATH / f'{prefix}_validation_backtest.png', dpi=150, bbox_inches='tight')
plt.show()
